<a href="https://colab.research.google.com/github/simona-wang/algorithms-for-massive-data/blob/main/linearProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Market Basket Analysis Project for Algorithm

#A-Priori Algorithm


In [15]:
#install Kaggle API
!pip install kaggle

#Import Libraries
import os
import pandas as pd
from itertools import combinations

#Kaggle authentication
os.environ['KAGGLE_USERNAME'] = "xxxxxx"
os.environ['KAGGLE_KEY'] = "xxxxxx"

!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows

!unzip imdb-dataset-of-top-1000-movies-and-tv-shows.zip

Dataset URL: https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
License(s): CC0-1.0
imdb-dataset-of-top-1000-movies-and-tv-shows.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  imdb-dataset-of-top-1000-movies-and-tv-shows.zip
replace imdb_top_1000.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [16]:
#Data Loading & Preprocessing
df = pd.read_csv("imdb_top_1000.csv")
print("Columns in the dataset:")
print(df.columns)
print()




Columns in the dataset:
Index(['Poster_Link', 'Series_Title', 'Released_Year', 'Certificate',
       'Runtime', 'Genre', 'IMDB_Rating', 'Overview', 'Meta_score', 'Director',
       'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross'],
      dtype='object')



The dataset was downloaded directly from Kaggle using the Kaggle API which ensures reproducibility of the experiments. The dataset contains metadata for the top 1000 IMDb movies and TV shows. For the market-basket analysis task, each movie is treated as a basket and the actors listed in the fiels Star 1,Star2, Star3, and Star4 are treated as items.

In [17]:
#select only the actor columns and store in a smaller table
actor_columns = ["Star1", "Star2", "Star3", "Star4"]
actors_df = df[actor_columns]
actors_df.head()


,Star1,Star2,Star3,Star4
0,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler
1,Marlon Brando,Al Pacino,James Caan,Diane Keaton
2,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine
3,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton
4,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler


I selected only the actor columns because the market-basket formulation uses the four stars of each movie as items. Reducing the DataFrame to these columns simplifies the preprocessing and makes the code clearer.

In [18]:
print("Missing values in actor columns:")
print(actors_df.isnull().sum())
print()
actors_df = actors_df.dropna()

Missing values in actor columns:
Star1    0
Star2    0
Star3    0
Star4    0
dtype: int64



The output tells us whether some movies are missing one or more actors. Since they are all 0s, we assume that the are no missing values. I checked for missing values because incomplete actor information would produce incomplete baskets and affect support counting.



In [19]:
# Building Baskets
baskets = []

for _, row in actors_df.iterrows():
    basket = {
        row["Star1"].strip(),
        row["Star2"].strip(),
        row["Star3"].strip(),
        row["Star4"].strip()
    }
    baskets.append(basket)

print(baskets[:5])
print("Number of baskets:", len(baskets))

[{'Bob Gunton', 'Morgan Freeman', 'Tim Robbins', 'William Sadler'}, {'Al Pacino', 'Marlon Brando', 'Diane Keaton', 'James Caan'}, {'Michael Caine', 'Heath Ledger', 'Aaron Eckhart', 'Christian Bale'}, {'Al Pacino', 'Robert Duvall', 'Robert De Niro', 'Diane Keaton'}, {'Henry Fonda', 'John Fiedler', 'Martin Balsam', 'Lee J. Cobb'}]
Number of baskets: 1000


##PASS 1 : Counting single actors

In [20]:
#we want to see how many movies contain each actor
actor_counts = {}

for basket in baskets:
    for actor in basket:
        if actor not in actor_counts:
            actor_counts[actor] = 1
        else:
            actor_counts[actor] += 1

#sorted the actor counts
sorted_actor_counts = sorted(actor_counts.items(), key=lambda x: x[1], reverse=True)

print("Top 10 actors by support count:")
for actor, count in sorted_actor_counts[:10]:
    print(actor, count)
print()

Top 10 actors by support count:
Robert De Niro 17
Tom Hanks 14
Al Pacino 13
Brad Pitt 12
Clint Eastwood 12
Christian Bale 11
Leonardo DiCaprio 11
Matt Damon 11
James Stewart 10
Michael Caine 9



I sorted the actor counts to inspect the most frequent actors in the dataset. This helped me understand the support distribution and choose an appropriate minimum support threshold.

###Support-Threshold choice
I tested several support thresholds empirically and compared the number of frequent singleton itemsets. This helped me select a threshold that balances meaningful results and computational efficiency.

In [21]:
supports = [0.003, 0.005, 0.01]

for s in supports:
    support_count = int(s * len(baskets))

    if support_count < 1:
        support_count = 1

    frequent_count = 0

    for actor, count in actor_counts.items():
        if count >= support_count:
            frequent_count += 1

    print("Support:", s)
    print("Support count threshold:", support_count)
    print("Number of frequent actors:", frequent_count)
    print()


Support: 0.003
Support count threshold: 3
Number of frequent actors: 271

Support: 0.005
Support count threshold: 5
Number of frequent actors: 79

Support: 0.01
Support count threshold: 10
Number of frequent actors: 9



I selected a minimum support of 0.003 after testing multiple thresholds and observing their effect on the number of frequent singleton itemsets. Higher thresholds were too restrictive for this dataset, since actors appear in relatively few movies. A threshold of 0.003, corresponding to a support count of about 3 transactions, provided a reasonable balance between keeping enough frequent actors for meaningful analysis and avoiding an excessively restrictive filtering.

In [22]:
min_support = 0.003
support_count = int(min_support * len(baskets))

if support_count < 1:
    support_count = 1

print("Chosen minimum support:", min_support)
print("Support count threshold:", support_count)
print()


Chosen minimum support: 0.003
Support count threshold: 3



In [23]:
#Find frequent singleton itemsets
#which actors are frequent ?
frequent_actors = set()
#keeps only actors whose support is large enough (>=3)
for actor, count in actor_counts.items():
    if count >= support_count:
        frequent_actors.add(actor)

print("Number of frequent actors:")
print(len(frequent_actors))
print()

#for each freqeunt actor, creates a tuple
frequent_actor_counts = []

for actor in frequent_actors:
    frequent_actor_counts.append((actor, actor_counts[actor]))

sorted_frequent_actors = sorted(
    frequent_actor_counts,
    key=lambda x: x[1],
    reverse=True
)

print("Top 10 frequent actors sorted by support count:")

for actor, count in sorted_frequent_actors[:10]:
    print(actor, count)


Number of frequent actors:
271

Top 10 frequent actors sorted by support count:
Robert De Niro 17
Tom Hanks 14
Al Pacino 13
Clint Eastwood 12
Brad Pitt 12
Leonardo DiCaprio 11
Matt Damon 11
Christian Bale 11
James Stewart 10
Humphrey Bogart 9


The support counts of singleton itemsets are computed in Pass 1 and stored in the dictionary actor_counts.

##PASS 2: Generating and counting actor/item pairs

now we want to find frequent pairs of actors (how many movies contain both actors) -> support count of a pair ;

We only consider pairs made of frequent actors because:
#"if an actor is not frequent, any pair cointaing it cannot be frequent"


In [24]:
pair_counts = {}

for basket in baskets:
    # Keep only the actors that were frequent in Pass 1
    filtered_basket = basket.intersection(frequent_actors)

    # Generate all 2-item combinations from the filtered basket
    for pair in combinations(filtered_basket, 2):
        # Sort the pair so that ('A', 'B') and ('B', 'A') are treated as the same pair
        pair = tuple(sorted(pair))

        if pair not in pair_counts:
            pair_counts[pair] = 1
        else:
            pair_counts[pair] += 1

#Extract frequent pairs-> we keep only the pairs whose support is high enough
frequent_pairs = {}

for pair, count in pair_counts.items():
    if count >= support_count:
        frequent_pairs[pair] = count

print("Number of frequent pairs:")
print(len(frequent_pairs))
print()

#sort frequent pairs by support count
sorted_pairs = sorted(frequent_pairs.items(), key=lambda x: x[1], reverse=True)

print("Top 10 frequent actor pairs:")
for pair, count in sorted_pairs[:10]:
    print(pair, count)


Number of frequent pairs:
25

Top 10 frequent actor pairs:
('Daniel Radcliffe', 'Rupert Grint') 6
('Daniel Radcliffe', 'Emma Watson') 5
('Emma Watson', 'Rupert Grint') 5
('Joe Pesci', 'Robert De Niro') 4
('Tim Allen', 'Tom Hanks') 4
('Al Pacino', 'Diane Keaton') 3
('Christian Bale', 'Michael Caine') 3
('Al Pacino', 'Robert De Niro') 3
('Ian McKellen', 'Orlando Bloom') 3
('Elijah Wood', 'Ian McKellen') 3


In the second pass of Apriori, I used the frequent singleton itemsets found in Pass 1 to generate candidate pairs. For each basket, I first removed actors that were not frequent, using the Apriori principle. Then I generated all 2-combinations of the remaining actors and counted their supports across the dataset. Finally, I filtered these candidate pairs by the minimum support threshold to obtain the frequent 2-itemsets.

In [25]:
#If we want to show the difference between all counted candidate pairs and the freqeunt ones
print("Number of candidate pairs counted:")
print(len(pair_counts))
print()

print("Number of frequent pairs:")
print(len(frequent_pairs))
print()

Number of candidate pairs counted:
622

Number of frequent pairs:
25



In [40]:
# APRIORI PASS 3 : just for fun :)
triple_counts = {}

for basket in baskets:

    filtered_basket = basket.intersection(frequent_actors)

    for triple in combinations(filtered_basket, 3):
        triple = tuple(sorted(triple))

        pair_subsets = list(combinations(triple, 2))

        # Check Apriori condition:
        # all 2-subsets of the triple must be frequent
        all_pairs_frequent = True
        for pair in pair_subsets:
            if pair not in frequent_pairs:
                all_pairs_frequent = False
                break

        if all_pairs_frequent:
            if triple not in triple_counts:
                triple_counts[triple] = 1
            else:
                triple_counts[triple] += 1



frequent_triples = {}

for triple, count in triple_counts.items():
    if count >= support_count:
        frequent_triples[triple] = count

print("Number of candidate triples counted:")
print(len(triple_counts))
print()

print("Number of frequent triples:")
print(len(frequent_triples))
print()

sorted_triples = sorted(frequent_triples.items(), key=lambda x: x[1], reverse=True)

print("Top x frequent actor triples:")
for triple, count in sorted_triples:
    print(triple, count)


Number of candidate triples counted:
4

Number of frequent triples:
3

Top x frequent actor triples:
('Daniel Radcliffe', 'Emma Watson', 'Rupert Grint') 5
('Elijah Wood', 'Ian McKellen', 'Orlando Bloom') 3
('Carrie Fisher', 'Harrison Ford', 'Mark Hamill') 3


#PCY Algorithm


##PCY pass 1 : count singletons supports and hash pair into buckets

In [27]:
#Buckets number choice
num_buckets = 6000

actor_counts = {}
bucket_counts = [0] * num_buckets

for basket in baskets:

    # Count singleton items
    for actor in basket:
        if actor not in actor_counts:
            actor_counts[actor] = 1
        else:
            actor_counts[actor] += 1

    # Hash pairs into buckets
    for pair in combinations(basket, 2):
        pair = tuple(sorted(pair))

        bucket_index = hash(pair) % num_buckets
        bucket_counts[bucket_index] += 1


In [28]:
#now we have a bitmap to check which buckets are frequent
frequent_buckets = [False] * num_buckets

for i in range(num_buckets):
    if bucket_counts[i] >= support_count:
        frequent_buckets[i] = True


*   True → bucket may contain frequent pairs;
*  False → bucket cannot contain frequent pairs;

If a bucket has count less than the support threshold then: no pair in that bucket can be frequent.



In the first pass of the PCY algorithm, I count singleton supports as in Apriori, but I also hash every pair of items into a fixed number of buckets. Each bucket stores how many pairs were hashed into it. After the pass, I identify which buckets are frequent by comparing their counts with the minimum support threshold.

##PCY pass 2: count only pairs whose items are frequent & whose bucket is frequent

In [29]:
#filter to keep only actors whose count is at least threshold
frequent_actors = set()

for actor, count in actor_counts.items():
    if count >= support_count:
        frequent_actors.add(actor)

print("Number of frequent actors:")
print(len(frequent_actors))
print()

#both conditions are met
pair_counts_pcy = {}

for basket in baskets:

    filtered_basket = basket.intersection(frequent_actors)

    for pair in combinations(filtered_basket, 2):
        pair = tuple(sorted(pair))

        bucket_index = hash(pair) % num_buckets

        # Count pair only if its bucket is frequent in pass 1
        if frequent_buckets[bucket_index]:
            if pair not in pair_counts_pcy:
                pair_counts_pcy[pair] = 1
            else:
                pair_counts_pcy[pair] += 1

#count the pairs
frequent_pairs_pcy = {}

for pair, count in pair_counts_pcy.items():
    if count >= support_count:
        frequent_pairs_pcy[pair] = count

print("Number of frequent pairs found by PCY:")
print(len(frequent_pairs_pcy))
print()

sorted_pairs_pcy = sorted(
    frequent_pairs_pcy.items(),
    key=lambda x: x[1],
    reverse=True
)

print("Top 10 frequent actor pairs found by PCY:")
for pair, count in sorted_pairs_pcy[:10]:
    print(pair, count)

Number of frequent actors:
271

Number of frequent pairs found by PCY:
25

Top 10 frequent actor pairs found by PCY:
('Daniel Radcliffe', 'Rupert Grint') 6
('Daniel Radcliffe', 'Emma Watson') 5
('Emma Watson', 'Rupert Grint') 5
('Joe Pesci', 'Robert De Niro') 4
('Tim Allen', 'Tom Hanks') 4
('Al Pacino', 'Diane Keaton') 3
('Christian Bale', 'Michael Caine') 3
('Al Pacino', 'Robert De Niro') 3
('Ian McKellen', 'Orlando Bloom') 3
('Elijah Wood', 'Ian McKellen') 3


In the second pass of PCY, I first restrict each basket to frequent singleton itemsets, as in Apriori. Then I generate candidate pairs, hash each pair to its bucket, and count the pair only if the corresponding bucket was marked as frequent in the first pass. This reduces the number of candidate pairs that need explicit counting.

The number of buckets in the PCY algorithm determines how pairs are distributed during the hashing step. If too few buckets are used, many pairs collide in the same bucket, causing most buckets to become frequent and reducing the pruning effect. Increasing the number of buckets reduces collisions and improves pruning. In our experiments, a value of 6000 buckets provided a noticeable reduction in candidate pairs.

In [30]:
#PCY diagnostics

print("Total number of buckets:")
print(num_buckets)
print()

print("Number of frequent buckets:")
print(sum(frequent_buckets))
print()

print("Apriori candidate pairs counted:")
print(len(pair_counts))
print()

print("PCY candidate pairs counted:")
print(len(pair_counts_pcy))
print()

reduction = (1 - len(pair_counts_pcy) / len(pair_counts)) * 100

print("Candidate pair reduction with PCY (%):")
print(round(reduction, 2), "%")

Total number of buckets:
6000

Number of frequent buckets:
496

Apriori candidate pairs counted:
622

PCY candidate pairs counted:
175

Candidate pair reduction with PCY (%):
71.86 %


To evaluate the effect of the PCY optimization, we compared the number of candidate pairs counted by Apriori and PCY. Using 6000 hash buckets, PCY reduced the number of candidate pairs by approximately 72%, demonstrating the pruning effect of the bucket-based filtering mechanism.

In [31]:
import time

#Scalability and Results Evaluation

In [32]:
comparison_table = pd.DataFrame({
    "Algorithm": ["Apriori", "PCY"],
    "Candidate Pairs": [
        len(pair_counts),
        len(pair_counts_pcy),
    ],
    "Frequent Pairs": [
        len(frequent_pairs),
        len(frequent_pairs_pcy)
    ]
})

print("Candidate Pair Comparison")
display(comparison_table)

Candidate Pair Comparison


,Algorithm,Candidate Pairs,Frequent Pairs
0,Apriori,622,25
1,PCY,175,25


In [33]:
print("PCY Diagnostics")
print()

print("Total number of buckets:")
print(num_buckets)
print()

print("Number of frequent buckets:")
print(sum(frequent_buckets))
print()

print("Apriori candidate pairs counted:")
print(len(pair_counts))
print()

print("PCY candidate pairs counted:")
print(len(pair_counts_pcy))
print()

reduction = (1 - len(pair_counts_pcy) / len(pair_counts)) * 100

print("Candidate pair reduction with PCY (%):")
print(round(reduction, 2), "%")
print()

print("Frequent bucket percentage:")
print(round((sum(frequent_buckets) / num_buckets) * 100, 2), "%")

PCY Diagnostics

Total number of buckets:
6000

Number of frequent buckets:
496

Apriori candidate pairs counted:
622

PCY candidate pairs counted:
175

Candidate pair reduction with PCY (%):
71.86 %

Frequent bucket percentage:
8.27 %


*Only 8.27% of the buckets are considered potentially frequent.*

##A-Priori vs PCY


To study scalability, Apriori and PCY are executed on increasing subsets of the basket dataset.

For each dataset size, the following values are recorded:

*   runtime
*   number of candidate pairs
*   number of frequent pairs

This allows us to observe how the algorithms behave as the number of baskets increases.

In [34]:
dataset_sizes = [200, 400, 600, 800, len(baskets)]

apriori_results = []
pcy_results = []

# APRIORI SCALABILITY EXPERIMENT

print("APRlORI SCALABILITY EXPERIMENT")
print()

for size in dataset_sizes:

    test_baskets = baskets[:size]

    local_support_count = int(min_support * len(test_baskets))
    if local_support_count < 1:
        local_support_count = 1

    start_time = time.time()

    # Pass 1: count actors
    actor_counts_test = {}

    for basket in test_baskets:
        for actor in basket:
            if actor not in actor_counts_test:
                actor_counts_test[actor] = 1
            else:
                actor_counts_test[actor] += 1

    # Find frequent actors
    frequent_actors_test = set()

    for actor, count in actor_counts_test.items():
        if count >= local_support_count:
            frequent_actors_test.add(actor)

    # Pass 2: count candidate pairs
    pair_counts_test = {}

    for basket in test_baskets:
        filtered_basket = basket.intersection(frequent_actors_test)

        for pair in combinations(filtered_basket, 2):
            pair = tuple(sorted(pair))

            if pair not in pair_counts_test:
                pair_counts_test[pair] = 1
            else:
                pair_counts_test[pair] += 1

    # Extract frequent pairs
    frequent_pairs_test = {}

    for pair, count in pair_counts_test.items():
        if count >= local_support_count:
            frequent_pairs_test[pair] = count

    end_time = time.time()
    runtime = end_time - start_time

    apriori_results.append((size, runtime, len(pair_counts_test), len(frequent_pairs_test), local_support_count))

    print("Dataset size:", size)
    print("Support threshold:", local_support_count)
    print("Runtime:", round(runtime, 4), "seconds")
    print("Candidate pairs:", len(pair_counts_test))
    print("Frequent pairs:", len(frequent_pairs_test))
    print()

# PCY SCALABILITY EXPERIMENT

print("PCY SCALABILITY EXPERIMENT")
print()

for size in dataset_sizes:

    test_baskets = baskets[:size]

    local_support_count = int(min_support * len(test_baskets))
    if local_support_count < 1:
        local_support_count = 1

    start_time = time.time()

    # PCY Pass 1
    actor_counts_test = {}
    bucket_counts_test = [0] * num_buckets

    for basket in test_baskets:

        # Count actors
        for actor in basket:
            if actor not in actor_counts_test:
                actor_counts_test[actor] = 1
            else:
                actor_counts_test[actor] += 1

        # Hash pairs into buckets
        for pair in combinations(basket, 2):
            pair = tuple(sorted(pair))
            bucket_index = hash(pair) % num_buckets
            bucket_counts_test[bucket_index] += 1

    # Frequent actors
    frequent_actors_test = set()

    for actor, count in actor_counts_test.items():
        if count >= local_support_count:
            frequent_actors_test.add(actor)

    # Frequent buckets
    frequent_buckets_test = [False] * num_buckets

    for i in range(num_buckets):
        if bucket_counts_test[i] >= local_support_count:
            frequent_buckets_test[i] = True

    # PCY Pass 2
    pair_counts_pcy_test = {}

    for basket in test_baskets:
        filtered_basket = basket.intersection(frequent_actors_test)

        for pair in combinations(filtered_basket, 2):
            pair = tuple(sorted(pair))
            bucket_index = hash(pair) % num_buckets

            if frequent_buckets_test[bucket_index]:
                if pair not in pair_counts_pcy_test:
                    pair_counts_pcy_test[pair] = 1
                else:
                    pair_counts_pcy_test[pair] += 1

    # Extract frequent pairs
    frequent_pairs_pcy_test = {}

    for pair, count in pair_counts_pcy_test.items():
        if count >= local_support_count:
            frequent_pairs_pcy_test[pair] = count

    end_time = time.time()
    runtime = end_time - start_time

    pcy_results.append((size, runtime, len(pair_counts_pcy_test), len(frequent_pairs_pcy_test), local_support_count))

    print("Dataset size:", size)
    print("Support threshold:", local_support_count)
    print("Runtime:", round(runtime, 4), "seconds")
    print("Candidate pairs:", len(pair_counts_pcy_test))
    print("Frequent pairs:", len(frequent_pairs_pcy_test))
    print()

APRlORI SCALABILITY EXPERIMENT

Dataset size: 200
Support threshold: 1
Runtime: 0.0017 seconds
Candidate pairs: 1162
Frequent pairs: 1162

Dataset size: 400
Support threshold: 1
Runtime: 0.0064 seconds
Candidate pairs: 2332
Frequent pairs: 2332

Dataset size: 600
Support threshold: 1
Runtime: 0.0111 seconds
Candidate pairs: 3503
Frequent pairs: 3503

Dataset size: 800
Support threshold: 2
Runtime: 0.0049 seconds
Candidate pairs: 1012
Frequent pairs: 101

Dataset size: 1000
Support threshold: 3
Runtime: 0.0049 seconds
Candidate pairs: 622
Frequent pairs: 25

PCY SCALABILITY EXPERIMENT

Dataset size: 200
Support threshold: 1
Runtime: 0.0077 seconds
Candidate pairs: 1162
Frequent pairs: 1162

Dataset size: 400
Support threshold: 1
Runtime: 0.0196 seconds
Candidate pairs: 2332
Frequent pairs: 2332

Dataset size: 600
Support threshold: 1
Runtime: 0.0354 seconds
Candidate pairs: 3503
Frequent pairs: 3503

Dataset size: 800
Support threshold: 2
Runtime: 0.0212 seconds
Candidate pairs: 569
Fre

In [35]:
scaling_table = pd.DataFrame({
    "Dataset Size": [r[0] for r in apriori_results],
    "Support Threshold": [r[4] for r in apriori_results],
    "Apriori Runtime (s)": [round(r[1], 4) for r in apriori_results],
    "Apriori Candidate Pairs": [r[2] for r in apriori_results],
    "Apriori Frequent Pairs": [r[3] for r in apriori_results],
    "PCY Runtime (s)": [round(r[1], 4) for r in pcy_results],
    "PCY Candidate Pairs": [r[2] for r in pcy_results],
    "PCY Frequent Pairs": [r[3] for r in pcy_results]
})

print("Scalability Experiment Results")
display(scaling_table)

Scalability Experiment Results


,Dataset Size,Support Threshold,Apriori Runtime (s),Apriori Candidate Pairs,Apriori Frequent Pairs,PCY Runtime (s),PCY Candidate Pairs,PCY Frequent Pairs
0,200,1,0.0017,1162,1162,0.0077,1162,1162
1,400,1,0.0064,2332,2332,0.0196,2332,2332
2,600,1,0.0111,3503,3503,0.0354,3503,3503
3,800,2,0.0049,1012,101,0.0212,569,101
4,1000,3,0.0049,622,25,0.0205,175,25


In [36]:
reduction_table = pd.DataFrame({
    "Dataset Size": [r[0] for r in apriori_results],
    "Apriori Candidates": [r[2] for r in apriori_results],
    "PCY Candidates": [r[2] for r in pcy_results],
    "Reduction (%)": [
        round((1 - pcy_results[i][2] / apriori_results[i][2]) * 100, 2)
        for i in range(len(apriori_results))
    ]
})

print("PCY Candidate Reduction % by Dataset Size")
display(reduction_table)

PCY Candidate Reduction % by Dataset Size


,Dataset Size,Apriori Candidates,PCY Candidates,Reduction (%)
0,200,1162,1162,0.00
1,400,2332,2332,0.00
2,600,3503,3503,0.00
3,800,1012,569,43.77
4,1000,622,175,71.86


In [38]:
print("Correctness Check")
print()

print("Apriori vs PCY identical:")
print(set(frequent_pairs.keys()) == set(frequent_pairs_pcy.keys()))
print()




Correctness Check

Apriori vs PCY identical:
True



#Discussion of Results

The three algorithms produced the same final frequent pairs, confirming the correctness of the implementations.

Apriori provides the baseline method for frequent itemset mining.  
PCY improves efficiency by reducing the number of candidate pairs through hash-based bucket pruning. In the full dataset experiment, PCY reduced the candidate pairs by approximately 65% compared to A-Priori.

The scalability experiment shows that, for small dataset sizes, the support threshold can become very low, making almost all observed pairs frequent. As the dataset size grows, the support threshold becomes more selective and the pruning effect of PCY becomes more visible.
